## Lab 6 : Knowledge reasoning with rule and KGE

In [15]:
import re
import json
import sys
from pathlib import Path
from typing import List, Tuple

import requests
from rdflib import Graph

In [16]:
TTL_FILE = "output/private_kb.ttl"

NT_FILE  = "output/expanded_kb.nt"

# Ollama configuration
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "gemma:2b" 


MAX_PREDICATES = 30
MAX_CLASSES    = 20
SAMPLE_TRIPLES = 15

MAX_REPAIR_ATTEMPTS = 2

In [17]:
def ask_local_llm(prompt: str, model: str = OLLAMA_MODEL) -> str:
    """
    Send a prompt to a local Ollama model and return the text response.
    Raises a clear error if Ollama is not running.
    """
    payload = {
        "model":  model,
        "prompt": prompt,
        "stream": False,   # disable streaming for simpler integration
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    except requests.exceptions.ConnectionError:
        raise RuntimeError(
            "\n[ERROR] Cannot connect to Ollama.\n"
            "  Make sure Ollama is running: ollama serve\n"
            f"  Expected URL: {OLLAMA_URL}"
        )

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama API error {response.status_code}: {response.text}"
        )

    data = response.json()
    return data.get("response", "").strip()


def check_ollama_running() -> bool:
    """Check if Ollama is reachable before starting."""
    try:
        r = requests.get("http://localhost:11434", timeout=5)
        return r.status_code == 200
    except Exception:
        return False


In [18]:
def load_graph(path: str) -> Graph:
    """
    Load the RDF knowledge graph.
    Tries Turtle first (better prefix support), falls back to N-Triples.
    """
    g = Graph()
    p = Path(path)

    if not p.exists():
        # Try N-Triples fallback
        nt = Path(NT_FILE)
        if nt.exists():
            print(f"  [{path}] not found — loading {NT_FILE} instead")
            g.parse(str(nt), format="nt")
            print(f"  Loaded {len(g):,} triples from {NT_FILE}")
            return g
        else:
            raise FileNotFoundError(
                f"Neither {path} nor {NT_FILE} found.\n"
                "Run Lab 1 (step1_build_rdf.py) first."
            )

    # Detect format from extension
    fmt = "turtle" if path.endswith(".ttl") else "nt"
    g.parse(str(p), format=fmt)
    print(f"  Loaded {len(g):,} triples from {path}")
    return g


In [19]:
def get_prefix_block(g: Graph) -> str:
    """
    Collect namespace prefixes from the graph.
    Always include the standard ones + our local AI-KB prefixes.
    """
    defaults = {
        "rdf":   "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs":  "http://www.w3.org/2000/01/rdf-schema#",
        "xsd":   "http://www.w3.org/2001/XMLSchema#",
        "owl":   "http://www.w3.org/2002/07/owl#",
        "local": "http://ai-kb.local/entity/",
        "onto":  "http://ai-kb.local/ontology/",
        "wd":    "http://www.wikidata.org/entity/",
        "wdt":   "http://www.wikidata.org/prop/direct/",
    }
    ns_map = {p: str(ns) for p, ns in g.namespace_manager.namespaces()}
    for k, v in defaults.items():
        ns_map.setdefault(k, v)
    lines = [f"PREFIX {p}: <{ns}>" for p, ns in sorted(ns_map.items())]
    return "\n".join(lines)


def list_distinct_predicates(g: Graph, limit: int = MAX_PREDICATES) -> List[str]:
    """List the most frequent predicates in the graph."""
    q = f"""
    SELECT DISTINCT ?p (COUNT(?p) AS ?cnt) WHERE {{
        ?s ?p ?o .
    }}
    GROUP BY ?p
    ORDER BY DESC(?cnt)
    LIMIT {limit}
    """
    try:
        return [str(row.p) for row in g.query(q)]
    except Exception:
        # Fallback without COUNT if not supported
        q2 = f"SELECT DISTINCT ?p WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
        return [str(row.p) for row in g.query(q2)]


def list_distinct_classes(g: Graph, limit: int = MAX_CLASSES) -> List[str]:
    """List the distinct RDF classes used in the graph."""
    q = f"""
    SELECT DISTINCT ?cls WHERE {{
        ?s a ?cls .
    }} LIMIT {limit}
    """
    return [str(row.cls) for row in g.query(q)]


def sample_triples(g: Graph, limit: int = SAMPLE_TRIPLES) -> List[Tuple]:
    """Sample representative triples to show the LLM the graph structure."""
    # Prefer triples with local entities (more informative than Wikidata QIDs)
    q = f"""
    SELECT ?s ?p ?o WHERE {{
        ?s ?p ?o .
        FILTER(STRSTARTS(STR(?s), "http://ai-kb.local/"))
    }} LIMIT {limit}
    """
    rows = [(str(r.s), str(r.p), str(r.o)) for r in g.query(q)]
    if len(rows) < limit:
        # Top up with any triples
        q2 = f"SELECT ?s ?p ?o WHERE {{ ?s ?p ?o . }} LIMIT {limit - len(rows)}"
        rows += [(str(r.s), str(r.p), str(r.o)) for r in g.query(q2)]
    return rows


def build_schema_summary(g: Graph) -> str:
    """
    Build a concise schema summary to inject into the LLM prompt.
    This helps the LLM generate valid SPARQL without hallucinating IRIs.
    """
    prefixes  = get_prefix_block(g)
    preds     = list_distinct_predicates(g)
    classes   = list_distinct_classes(g)
    samples   = sample_triples(g)

    pred_lines   = "\n".join(f"  - {p}" for p in preds)
    class_lines  = "\n".join(f"  - {c}" for c in classes)
    sample_lines = "\n".join(f"  - <{s}> <{p}> <{o}>" for s, p, o in samples)

    summary = f"""
{prefixes}

# Domain: AI Research (researchers, organizations, awards, publications)
# Key entity namespace: http://ai-kb.local/entity/<CamelCaseName>
# Example entities: local:DarioAmodei, local:IlyaSutskever, local:GoogleResearch
# Wikidata entities: wd:Q... (expanded via SPARQL in Lab 2)

# Predicates available (most frequent first):
{pred_lines}

# Classes / rdf:type values:
{class_lines}

# Sample triples (showing structure):
{sample_lines}
"""
    return summary.strip()

In [20]:
SPARQL_INSTRUCTIONS = """You are a SPARQL 1.1 generator for an AI Research knowledge graph.
Convert the user QUESTION into a valid SPARQL SELECT query.

STRICT RULES:
- Use ONLY these prefixes:
  PREFIX local: <http://ai-kb.local/entity/>
  PREFIX onto: <http://ai-kb.local/ontology/>
  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
  PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
- The main relation between entities is: onto:relatedTo
- Entity names are CamelCase: local:DarioAmodei, local:TomBrown, local:GoogleResearch
- To find relations of an entity use: ?s onto:relatedTo ?o
- NEVER use wdt:, foaf:, or any other prefix not listed above.
- Return ONLY the SPARQL query inside a single ```sparql code block.
- No explanations outside the code block.

EXAMPLE valid query:
```sparql
SELECT ?related WHERE {
  local:DarioAmodei onto:relatedTo ?related .
}
```
"""


def make_sparql_prompt(schema_summary: str, question: str) -> str:
    return f"""{SPARQL_INSTRUCTIONS}

SCHEMA SUMMARY:
{schema_summary}

QUESTION:
{question}

Return only the SPARQL query in a ```sparql code block."""


CODE_BLOCK_RE = re.compile(r"```(?:sparql)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)


def extract_sparql(text: str) -> str:
    """Extract SPARQL from a code block, or return the raw text as fallback."""
    m = CODE_BLOCK_RE.search(text)
    if m:
        return m.group(1).strip()
    # Try to extract anything that looks like a SELECT query
    select_match = re.search(r"(SELECT\s.*)", text, re.IGNORECASE | re.DOTALL)
    if select_match:
        return select_match.group(1).strip()
    return text.strip()


def generate_sparql(question: str, schema_summary: str) -> str:
    """Ask the LLM to generate SPARQL from a natural language question."""
    raw   = ask_local_llm(make_sparql_prompt(schema_summary, question))
    query = extract_sparql(raw)
    return query

In [21]:
def run_sparql(g: Graph, query: str) -> Tuple[List[str], List[Tuple]]:
    """Execute a SPARQL query on the graph and return (variables, rows)."""
    result = g.query(query)
    vars_  = [str(v) for v in result.vars]
    rows   = [tuple(str(cell) if cell else "" for cell in r) for r in result]
    return vars_, rows


REPAIR_PROMPT_TEMPLATE = """The SPARQL query below failed to execute on an AI Research RDF graph.
Using the SCHEMA SUMMARY and the ERROR MESSAGE, return a corrected SPARQL 1.1 SELECT query.

Rules:
- Use only known prefixes and IRIs from the schema.
- Keep the query as simple and robust as possible.
- Return only the corrected SPARQL in a single ```sparql code block.

SCHEMA SUMMARY:
{schema}

ORIGINAL QUESTION:
{question}

FAILED SPARQL:
{bad_query}

ERROR MESSAGE:
{error}

Return only the corrected SPARQL in a ```sparql code block."""


def repair_sparql(schema_summary: str, question: str,
                  bad_query: str, error_msg: str) -> str:
    """Ask the LLM to fix a broken SPARQL query."""
    prompt = REPAIR_PROMPT_TEMPLATE.format(
        schema=schema_summary,
        question=question,
        bad_query=bad_query,
        error=error_msg,
    )
    raw = ask_local_llm(prompt)
    return extract_sparql(raw)


def answer_with_rag(g: Graph, schema_summary: str, question: str,
                    try_repair: bool = True) -> dict:
    """
    Full RAG pipeline:
      1. Generate SPARQL from question
      2. Execute SPARQL on the graph
      3. If it fails, attempt self-repair (up to MAX_REPAIR_ATTEMPTS times)

    Returns a dict with keys:
      query, vars, rows, repaired, repair_attempts, error
    """
    sparql = generate_sparql(question, schema_summary)

    for attempt in range(MAX_REPAIR_ATTEMPTS + 1):
        try:
            vars_, rows = run_sparql(g, sparql)
            return {
                "query":           sparql,
                "vars":            vars_,
                "rows":            rows,
                "repaired":        attempt > 0,
                "repair_attempts": attempt,
                "error":           None,
            }
        except Exception as e:
            err = str(e)
            if try_repair and attempt < MAX_REPAIR_ATTEMPTS:
                print(f"  [Repair attempt {attempt+1}/{MAX_REPAIR_ATTEMPTS}] {err[:80]}...")
                sparql = repair_sparql(schema_summary, question, sparql, err)
            else:
                return {
                    "query":           sparql,
                    "vars":            [],
                    "rows":            [],
                    "repaired":        attempt > 0,
                    "repair_attempts": attempt,
                    "error":           err,
                }


In [22]:
def answer_no_rag(question: str) -> str:
    """
    Baseline: ask the LLM directly without any KB context.
    Used to demonstrate the value of RAG.
    """
    prompt = (
        f"Answer the following question about AI research as best you can.\n"
        f"If you don't know the answer, say so honestly.\n\n"
        f"Question: {question}"
    )
    return ask_local_llm(prompt)

In [23]:
def pretty_print_rag_result(result: dict, max_rows: int = 20):
    """Pretty-print the RAG pipeline result."""
    print("\n  [SPARQL Query Generated]")
    print("  " + "-" * 50)
    for line in result["query"].split("\n"):
        print(f"  {line}")
    print("  " + "-" * 50)

    if result.get("repaired"):
        print(f"  [Self-repair: {result['repair_attempts']} attempt(s)]")

    if result.get("error"):
        print(f"\n  [ERROR] Query failed: {result['error'][:200]}")
        return

    vars_ = result.get("vars", [])
    rows  = result.get("rows", [])

    if not rows:
        print("\n  [No results found in the knowledge graph]")
        return

    print(f"\n  [Results — {len(rows)} row(s)]")
    print("  " + " | ".join(f"{v:30s}" for v in vars_))
    print("  " + "-" * (32 * len(vars_)))
    for row in rows[:max_rows]:
        # Shorten long URIs for display
        display = []
        for cell in row:
            if cell.startswith("http://ai-kb.local/entity/"):
                display.append(cell.replace("http://ai-kb.local/entity/", "local:"))
            elif cell.startswith("http://www.wikidata.org/entity/"):
                display.append(cell.replace("http://www.wikidata.org/entity/", "wd:"))
            elif cell.startswith("http://www.wikidata.org/prop/direct/"):
                display.append(cell.replace("http://www.wikidata.org/prop/direct/", "wdt:"))
            else:
                display.append(cell[:50])
        print("  " + " | ".join(f"{d:30s}" for d in display))

    if len(rows) > max_rows:
        print(f"  ... (showing {max_rows} of {len(rows)} rows)")


In [24]:
EVALUATION_QUESTIONS = [
    "Who is related to Dario Amodei in the knowledge graph?",
    "Which persons are affiliated with Google Research?",
    "What organizations are present in the knowledge graph?",
    "Who is Ilya Sutskever related to?",
    "List all persons who are related to OpenAI.",
]


def run_evaluation(g: Graph, schema_summary: str):
    """
    Run the 5 evaluation questions and print a comparison table
    (baseline vs RAG) for the report.
    """
    print("\n" + "=" * 70)
    print("  EVALUATION — Baseline vs SPARQL-generation RAG")
    print("  (5 questions on the AI Research Knowledge Graph)")
    print("=" * 70)

    results = []

    for i, question in enumerate(EVALUATION_QUESTIONS, 1):
        print(f"\n[Q{i}] {question}")
        print("-" * 60)

        # Baseline
        print("  Baseline (no RAG):")
        baseline = answer_no_rag(question)
        print(f"  {baseline[:300]}{'...' if len(baseline) > 300 else ''}")

        # RAG
        print("\n  RAG (SPARQL generation):")
        rag_result = answer_with_rag(g, schema_summary, question)
        pretty_print_rag_result(rag_result)

        results.append({
            "question":        question,
            "baseline":        baseline[:200],
            "rag_rows":        len(rag_result.get("rows", [])),
            "rag_error":       rag_result.get("error"),
            "rag_repaired":    rag_result.get("repaired"),
            "sparql":          rag_result.get("query", ""),
        })

    # Summary table
    print("\n" + "=" * 70)
    print("  SUMMARY TABLE")
    print("=" * 70)
    print(f"  {'Q':3s} {'Results':>8s} {'Repaired':>10s} {'Error':>6s}")
    print("  " + "-" * 35)
    for i, r in enumerate(results, 1):
        n_rows   = r["rag_rows"]
        repaired = "yes" if r["rag_repaired"] else "no"
        error    = "yes" if r["rag_error"] else "no"
        print(f"  Q{i:<2d} {n_rows:>8d} {repaired:>10s} {error:>6s}")

    return results


In [25]:
def run_cli(g: Graph, schema_summary: str):
    """
    Interactive CLI loop: ask questions and get RAG-grounded answers.
    Type 'eval' to run the 5 evaluation questions.
    Type 'schema' to print the schema summary.
    Type 'quit' to exit.
    """
    print("\n" + "=" * 60)
    print("  RAG Chatbot — AI Research Knowledge Graph")
    print(f"  Model: {OLLAMA_MODEL} | Graph: {TTL_FILE}")
    print("  Commands: 'eval' | 'schema' | 'quit'")
    print("=" * 60)

    while True:
        try:
            q = input("\nQuestion: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye!")
            break

        if not q:
            continue

        if q.lower() == "quit":
            print("Bye!")
            break

        if q.lower() == "eval":
            run_evaluation(g, schema_summary)
            continue

        if q.lower() == "schema":
            print("\n[Schema Summary]")
            print(schema_summary)
            continue

        # Normal question
        print("\n--- Baseline (No RAG) ---")
        print(answer_no_rag(q))

        print("\n--- SPARQL-generation RAG ---")
        result = answer_with_rag(g, schema_summary, q, try_repair=True)
        pretty_print_rag_result(result)



In [26]:
#Main
if __name__ == "__main__":
    print("=" * 60)
    print("  RAG Lab — AI Research Knowledge Graph")
    print("=" * 60)

    # Check Ollama
    print("\n[0] Checking Ollama...")
    if not check_ollama_running():
        print("  [ERROR] Ollama is not running.")
        print("  Start it with: ollama serve")
        print("  Then pull a model: ollama pull gemma:2b")
        sys.exit(1)
    print(f"  Ollama is running. Model: {OLLAMA_MODEL}")

    # Load graph
    print("\n[1] Loading knowledge graph...")
    g = load_graph(TTL_FILE)

    # Build schema summary
    print("\n[2] Building schema summary...")
    schema = build_schema_summary(g)
    print(f"  Schema summary built ({len(schema)} chars)")
    print("  Preview:")
    for line in schema.split("\n")[:10]:
        print(f"    {line}")
    print("    ...")

    # Check for --eval flag
    if "--eval" in sys.argv:
        run_evaluation(g, schema)
    else:
        # Interactive CLI
        run_cli(g, schema)


  RAG Lab — AI Research Knowledge Graph

[0] Checking Ollama...
  Ollama is running. Model: gemma:2b

[1] Loading knowledge graph...
  Loaded 2,246 triples from output/private_kb.ttl

[2] Building schema summary...
  Schema summary built (4366 chars)
  Preview:
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX csvw: <http://www.w3.org/ns/csvw#>
    PREFIX dc: <http://purl.org/dc/elements/1.1/>
    PREFIX dcam: <http://purl.org/dc/dcam/>
    PREFIX dcat: <http://www.w3.org/ns/dcat#>
    PREFIX dcmitype: <http://purl.org/dc/dcmitype/>
    PREFIX dcterms: <http://purl.org/dc/terms/>
    PREFIX doap: <http://usefulinc.com/ns/doap#>
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX geo: <http://www.opengis.net/ont/geosparql#>
    ...

  RAG Chatbot — AI Research Knowledge Graph
  Model: gemma:2b | Graph: output/private_kb.ttl
  Commands: 'eval' | 'schema' | 'quit'



Question:  Who is related to Dario Amodei?



--- Baseline (No RAG) ---
I am unable to access external sources or provide real-time information, so I cannot answer this question.

--- SPARQL-generation RAG ---

  [SPARQL Query Generated]
  --------------------------------------------------
  SELECT ?related WHERE {
    local:DarioAmodei onto:relatedTo ?related .
  }
  --------------------------------------------------

  [Results — 1 row(s)]
  related                       
  --------------------------------
  local:TomBrown                



Question:  Which persons are affiliated with Google Research?



--- Baseline (No RAG) ---
I am unable to provide specific details about individuals affiliated with Google Research, as I do not have access to real-time or specific personnel information.

--- SPARQL-generation RAG ---

  [SPARQL Query Generated]
  --------------------------------------------------
  SELECT ?person WHERE {
    ?person <http://ai-kb.local/ontology/relatedTo> <http://ai-kb.local/entity/GoogleResearch>.
  }
  --------------------------------------------------

  [Results — 1 row(s)]
  person                        
  --------------------------------
  local:AdamPearce              


...


--- Baseline (No RAG) ---
I am unable to access external knowledge sources, including the Knowledge Graph, and cannot provide a list of organizations or entities in the knowledge graph.

--- SPARQL-generation RAG ---

  [SPARQL Query Generated]
  --------------------------------------------------
  SELECT DISTINCT ?organization WHERE {
    ?organization onto:relatedTo ?related .
  }
  --------------------------------------------------

  [Results — 352 row(s)]
  organization                  
  --------------------------------
  local:AdamPearce              
  local:Akkaya                  
  local:AnirbanSantara          
  local:Arxiv                   
  local:CaricaturesCammerata    
  local:Alec                    
  local:Gabriel                 
  local:Mordvintsev             
  local:Petrov                  
  local:AshishTendulkar         
  local:AnshKhurana             
  local:Atari                   
  local:Athalye                 
  local:D                       
  lo


Question:  Who is related to Tom Brown?



--- Baseline (No RAG) ---
I am unable to provide specific details about people or their relationships, as I do not have access to real-time or personal information.

--- SPARQL-generation RAG ---

  [SPARQL Query Generated]
  --------------------------------------------------
  SELECT ?related WHERE {
    local:TomBrown onto:relatedTo ?related .
  }
  --------------------------------------------------

  [Results — 1 row(s)]
  related                       
  --------------------------------
  local:JeffClune               



Question:  eval



  EVALUATION — Baseline vs SPARQL-generation RAG
  (5 questions on the AI Research Knowledge Graph)

[Q1] Who is related to Dario Amodei in the knowledge graph?
------------------------------------------------------------
  Baseline (no RAG):
  I cannot access external sources or provide information about specific individuals or entities, including Dario Amodei. Therefore, I cannot answer this question from the provided context.

  RAG (SPARQL generation):

  [SPARQL Query Generated]
  --------------------------------------------------
  SELECT ?related WHERE {
    local:DarioAmodei onto:relatedTo ?related .
  }
  --------------------------------------------------

  [Results — 1 row(s)]
  related                       
  --------------------------------
  local:TomBrown                

[Q2] Which persons are affiliated with Google Research?
------------------------------------------------------------
  Baseline (no RAG):
  I am unable to provide specific personal information about i